# Combined reporter titration

Low-range titration curves (cells per guide ≤ 210) of distinctiveness and EBI mAP for the combined-reporter aggregates: cp (7 reporters combined) and live-cell-matched (7 reporters combined).

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Combined-reporter aggregate titration CSV (long form: `group`, `x`, `metric`, `y`) curated into `../../data/figures/figure_4/` (see README).

In [ ]:
FIGURE_DATA = Path("../../data/figures/figure_4")

COMBINED_CSV = FIGURE_DATA / "combined_reporter_titration.csv"

## Load and preprocess

Pivot the combined-reporter CSV from long to wide and split it into one dataframe per modality.

In [ ]:
GROUP_REMAP = {
    "cp":                    "cp",
    "matched_livecell_best": "live-cell",
}

_cdf = pd.read_csv(COMBINED_CSV)
_cdf = _cdf[_cdf["group"].isin(GROUP_REMAP)]
combined = {}
for group, grp in _cdf.groupby("group"):
    pivot = grp.pivot(index="x", columns="metric", values="y").reset_index()
    pivot = pivot.rename(columns={
        "x":               "cells_per_guide",
        "distinctiveness": "distinctiveness_map_mean",
        "ebi":             "ebi_map_mean",
        "activity":        "activity_map_mean",
        "corum":           "corum_map_mean",
    })
    combined[GROUP_REMAP[group]] = pivot.sort_values("cells_per_guide")

## Shared plot styling

Per-modality color and legend label used across both plots.

In [ ]:
COMBINED_COLORS = {"cp": "#7b5ea7", "live-cell": "#555555"}

COMBINED_LABELS = {
    "cp":        "cp (7 reporters combined)",
    "live-cell": "live-cell-matched (7 reporters combined)",
}

## Plot 1 — Distinctiveness, combined-reporter aggregates (low range)

Combined-reporter distinctiveness curves for cp and live-cell-matched at low cells-per-guide.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2))

for modality, df_c in combined.items():
    ax.plot(
        df_c["cells_per_guide"],
        df_c["distinctiveness_map_mean"],
        color=COMBINED_COLORS[modality],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label=COMBINED_LABELS[modality],
        zorder=5,
    )

ax.set_xlim(1, 210)
ax.set_ylim(0, 0.3)
ax.set_yticks(np.arange(0, 0.31, 0.1))
ax.set_xlabel("Cells per guide")
ax.set_ylabel("Distinctiveness mAP")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_distinctiveness_combined.svg", bbox_inches="tight")
plt.show()

## Plot 2 — EBI, combined-reporter aggregates (low range)

Same low-range view as plot 1 but for EBI mAP.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2))

for modality, df_c in combined.items():
    ax.plot(
        df_c["cells_per_guide"],
        df_c["ebi_map_mean"],
        color=COMBINED_COLORS[modality],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label=COMBINED_LABELS[modality],
        zorder=5,
    )

ax.set_xlim(1, 210)
ax.set_ylim(0, 0.55)
ax.set_xlabel("Cells per guide")
ax.set_ylabel("EBI mAP")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_ebi_combined.svg", bbox_inches="tight")
plt.show()